# 04 — Synchronised Protein Movies and Audio

This notebook builds the deliverables for the study. It consumes:

- the aligned trajectories (raw `.xtc` files in `data/raw/`)
- the per-state sonification mapping tables (`data/processed/{state}_sonification_mapping.csv`, from notebook 02)
- the per-state per-instrument WAV files (`outputs/audio/`, from notebook 02)

and produces, under `outputs/video/`:

| Deliverable                                  | What it is                                                                                   |
|----------------------------------------------|----------------------------------------------------------------------------------------------|
| `mov_{state}_silent.mp4`                     | One silent protein movie per state, in cartoon representation with key motifs colour-coded.  |
| `mov_{state}_{instrument}.mp4`               | The same movie with one of the three audio renders baked in. **Drop straight into a slide.** |
| `mov_pair_{pair}_{instrument}.mp4`           | Two protein panels side-by-side, sequential A → silence → B, with the pairwise audio.        |
| `presentation.html`                          | Interactive HTML page: two panels, three instrument audio bars under each, click to play.    |

### Synchronisation principle

Each video frame is held on screen for *exactly* the duration of the
corresponding MIDI note (plus the inter-note gap). Since both the audio
and the video derive from the same mapping table, the protein
geometry on screen always matches the sound being heard, frame by frame.

### Time budget

PyMOL ray-tracing is CPU-bound and can take ~0.5-2 s per frame at
480 × 480 pixels. Set `MAX_FRAMES` to a small number (e.g. 50) for a
quick end-to-end test before launching a full render.


## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Install dependencies

PyMOL is installed via `apt`. `moviepy` is the higher-level video editor.
`Xvfb` provides a virtual display so PyMOL's OpenGL renderer works in
the headless Colab VM (ray-tracing also works without it, but having
a display means we can fall back to fast non-ray rendering for tests).


In [ ]:
# Install ONLY what's missing on a fresh Colab. Restart the runtime
# first (Runtime -> Restart session) if you are re-running this cell.
!apt-get -qq update
!apt-get -qq install -y xvfb ffmpeg
!pip -q install "MDAnalysis>=2.8.0" moviepy==1.0.3 imageio-ffmpeg "imageio[pyav]" pillow pymol-open-source
# Start a virtual display for OpenGL-based rendering (used when raytrace=False)
import subprocess, os
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1024x768x24'])
os.environ['DISPLAY'] = ':99'

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 2.1 MB/s eta 0:00:00


## 3. Imports, paths, and configuration


In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.analysis import align

PROJECT_DIR = Path('/content/drive/MyDrive/GPCR_Sonification')
RAW_DIR        = PROJECT_DIR / 'data' / 'raw'
PROCESSED_DIR  = PROJECT_DIR / 'data' / 'processed'
AUDIO_DIR      = PROJECT_DIR / 'outputs' / 'audio'
FIG_DIR        = PROJECT_DIR / 'outputs' / 'figures'
VIDEO_DIR      = PROJECT_DIR / 'outputs' / 'video'
FRAMES_DIR     = VIDEO_DIR / 'frames'
ALIGNED_DIR    = VIDEO_DIR / 'aligned'
for d in [VIDEO_DIR, FRAMES_DIR, ALIGNED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Trajectory stride. MUST match the stride used in notebook 01, because
# the per-state mapping CSVs (which set per-frame note durations) were
# built at that stride. If you change FRAME_STRIDE here, regenerate
# notebooks 01 and 02 with the same value.
FRAME_STRIDE = 5

# Rendering knobs
RENDER_W, RENDER_H = 480, 480       # per-panel pixel size
RAYTRACE = True                      # True = pretty, slow (CPU). False = fast, needs Xvfb.
ANTIALIAS = 2                        # PyMOL antialias (0-3); 2 is a good middle ground
NOTE_GAP_S = 0.02                    # must match notebook 02
PAIR_SILENCE_S = 1.5                 # must match notebook 02
MAX_FRAMES = None                    # int -> cap frames per state for testing; None -> use all

# The three systems
SYSTEMS = {
    'inactive':            {'has_ligand': False},
    'active':              {'has_ligand': False},
    'active_ligand_bound': {'has_ligand': True},
}
for s in SYSTEMS:
    SYSTEMS[s]['topology']   = RAW_DIR / s / 'topology.pdb'
    SYSTEMS[s]['trajectory'] = RAW_DIR / s / 'trajectory.xtc'

# The two pairwise comparisons
PAIRS = {
    'activation': {'states': ['inactive', 'active'],
                   'label':  'Activation: inactive vs active'},
    'ligand':     {'states': ['active', 'active_ligand_bound'],
                   'label':  'Ligand binding: active apo vs active + ligand'},
}

INSTRUMENTS = ['piano', 'violin', 'flute']

print('Configuration:')
print(f'  raytrace      = {RAYTRACE}')
print(f'  render size   = {RENDER_W} x {RENDER_H}')
print(f'  frame stride  = {FRAME_STRIDE} (must match notebook 01/02)')
print(f'  max frames    = {MAX_FRAMES}')
print(f'  systems       = {list(SYSTEMS)}')
print(f'  pairs         = {list(PAIRS)}')


Configuration:
  raytrace      = True
  render size   = 480 x 480
  frame stride  = 5 (must match notebook 01/02)
  max frames    = None
  systems       = ['inactive', 'active', 'active_ligand_bound']
  pairs         = ['activation', 'ligand']


## 4. Trajectory preprocessing for PyMOL

PyMOL does not read `.xtc` directly. We use MDAnalysis to:

1. Align each trajectory in memory on all Cα atoms (so the protein
   does not tumble around in the video).
2. Apply the same `FRAME_STRIDE` used in notebooks 01 and 02 so that
   one trajectory frame in this notebook = one note in the sonification.
3. Write the strided, aligned trajectory as a `.dcd` file and the
   reference structure as a single-frame `.pdb`. PyMOL reads both.


In [ ]:
def preprocess_trajectory(state, cfg, aligned_dir, stride=FRAME_STRIDE,
                          max_frames=MAX_FRAMES):
    print(f'\n[{state}] preprocessing for PyMOL')
    if not cfg['topology'].exists() or not cfg['trajectory'].exists():
        print(f'  [SKIP] missing files for {state}')
        return None, None, 0

    u = mda.Universe(str(cfg['topology']), str(cfg['trajectory']))
    print(f'  raw frames: {len(u.trajectory)}')

    print('  aligning on Cα ...')
    align.AlignTraj(u, u, select='protein and name CA',
                    in_memory=True, verbose=False).run()

    # Decide which frames to keep
    indices = list(range(0, len(u.trajectory), stride))
    if max_frames is not None:
        indices = indices[:max_frames]
    print(f'  keeping {len(indices)} frames (stride={stride}, max={max_frames})')

    # Reference (first frame after alignment) as PDB
    pdb_path = aligned_dir / f'{state}_ref.pdb'
    u.trajectory[0]
    with mda.Writer(str(pdb_path), u.atoms.n_atoms) as W:
        W.write(u.atoms)

    # Strided trajectory as DCD
    dcd_path = aligned_dir / f'{state}_aligned.dcd'
    with mda.Writer(str(dcd_path), u.atoms.n_atoms) as W:
        for idx in indices:
            u.trajectory[idx]
            W.write(u.atoms)

    print(f'  wrote {pdb_path.name}, {dcd_path.name}')
    return pdb_path, dcd_path, len(indices)

state_frame_counts = {}
for state, cfg in SYSTEMS.items():
    pdb, dcd, n = preprocess_trajectory(state, cfg, ALIGNED_DIR)
    state_frame_counts[state] = n
print('\nFrame counts per state:', state_frame_counts)



[inactive] preprocessing for PyMOL
  raw frames: 2500
  aligning on Cα ...
  keeping 500 frames (stride=5, max=None)


/usr/local/lib/python3.12/dist-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


  wrote inactive_ref.pdb, inactive_aligned.dcd

[active] preprocessing for PyMOL
  raw frames: 2500
  aligning on Cα ...
  keeping 500 frames (stride=5, max=None)
  wrote active_ref.pdb, active_aligned.dcd

[active_ligand_bound] preprocessing for PyMOL
  raw frames: 2500
  aligning on Cα ...
  keeping 500 frames (stride=5, max=None)
  wrote active_ligand_bound_ref.pdb, active_ligand_bound_aligned.dcd

Frame counts per state: {'inactive': 500, 'active': 500, 'active_ligand_bound': 500}


## 5. Render frames with PyMOL

For each state we render `N` PNG frames at `RENDER_W × RENDER_H`.

### Visualisation rules

- Whole protein: cartoon, sky-blue.
- TM3 intracellular (resid 128-135): crimson.
- TM6 intracellular (resid 265-275): orange.
- NPxxY motif (resid 318-328): purple.
- DRY ionic-lock pair (R3.50, E6.30): shown as sticks, yellow.
- Ligand (if present): shown as green sticks.

### Speed knobs

- `RAYTRACE = True` produces clean, publication-quality frames but is
  slow (~0.5-2 s per frame at 480 × 480). For a 200-frame trajectory ×
  3 states this is ~10-20 min on a free-tier Colab CPU.
- `RAYTRACE = False` uses OpenGL through the Xvfb virtual display set
  up in cell 2; ~10× faster but visibly less smooth.
- Drop `RENDER_W/RENDER_H` to 360 for an even faster first pass.


In [ ]:
import pymol
pymol.finish_launching(['pymol', '-Qcq'])  # quiet, command-line, no GUI
from pymol import cmd

def style_for_state(has_ligand):
    cmd.bg_color('white')
    cmd.hide('everything')
    cmd.show('cartoon', 'polymer')
    cmd.set('cartoon_transparency', 0.0)
    cmd.color('skyblue', 'polymer')
    # Region highlights
    cmd.color('red', 'polymer and resi 128-135') # Changed 'crimson' to 'red'
    cmd.color('orange',  'polymer and resi 265-275')
    cmd.color('purple',  'polymer and resi 318-328')
    # Ionic-lock pair as sticks
    cmd.show('sticks', '(polymer and resi 131) or (polymer and resi 268)')
    cmd.color('yellow', '(polymer and resi 131) or (polymer and resi 268)')
    cmd.set('stick_radius', 0.18)
    # Ligand if present
    if has_ligand:
        # Changed the selection string to be more explicit with 'or' conditions
        sel = ('not polymer and '
               'not (resname POPC or resname POPE or resname POPS or resname POPG or resname CHL1 or resname CHOL) and '
               'not (resname TIP3 or resname SOL or resname WAT or resname HOH) and '
               'not (resname NA or resname CL or resname K or resname MG or resname CA or resname ZN)')
        cmd.show('sticks', sel)
        cmd.color('forest', sel)
    # Render quality
    cmd.set('antialias', ANTIALIAS)
    cmd.set('ray_opaque_background', 1)
    cmd.set('ambient', 0.30)
    cmd.set('ray_shadows', 0)  # faster

def render_state(state, cfg, pdb_path, dcd_path, frames_subdir):
    if pdb_path is None or dcd_path is None:
        return
    print(f'\n[{state}] rendering frames -> {frames_subdir}')
    cmd.reinitialize()
    cmd.load(str(pdb_path), 'prot')
    cmd.load_traj(str(dcd_path), 'prot')
    style_for_state(cfg['has_ligand'])
    cmd.orient('polymer')
    cmd.zoom('polymer', buffer=3)

    n_frames = cmd.count_states('prot')
    if MAX_FRAMES is not None:
        n_frames = min(n_frames, MAX_FRAMES)
    frames_subdir.mkdir(parents=True, exist_ok=True)
    for i in range(1, n_frames + 1):
        cmd.frame(i)
        out_png = frames_subdir / f'frame_{i:04d}.png'
        cmd.png(str(out_png), width=RENDER_W, height=RENDER_H,
                ray=int(RAYTRACE), dpi=100)
        if i % 25 == 0 or i == n_frames:
            print(f'  rendered {i}/{n_frames}')
    print(f'[{state}] done ({n_frames} frames).')

for state, cfg in SYSTEMS.items():
    pdb = ALIGNED_DIR / f'{state}_ref.pdb'
    dcd = ALIGNED_DIR / f'{state}_aligned.dcd'
    if not (pdb.exists() and dcd.exists()):
        print(f'[SKIP] {state}: aligned files missing')
        continue
    render_state(state, cfg, pdb, dcd, FRAMES_DIR / state)



[inactive] rendering frames -> /content/drive/MyDrive/GPCR_Sonification/outputs/video/frames/inactive
  rendered 25/500
  rendered 50/500
  rendered 75/500
  rendered 100/500
  rendered 125/500
  rendered 150/500
  rendered 175/500
  rendered 200/500
  rendered 225/500
  rendered 250/500
  rendered 275/500
  rendered 300/500
  rendered 325/500
  rendered 350/500
  rendered 375/500
  rendered 400/500
  rendered 425/500
  rendered 450/500
  rendered 475/500
  rendered 500/500
[inactive] done (500 frames).

[active] rendering frames -> /content/drive/MyDrive/GPCR_Sonification/outputs/video/frames/active
  rendered 25/500
  rendered 50/500
  rendered 75/500
  rendered 100/500
  rendered 125/500
  rendered 150/500
  rendered 175/500
  rendered 200/500
  rendered 225/500
  rendered 250/500
  rendered 275/500
  rendered 300/500
  rendered 325/500
  rendered 350/500
  rendered 375/500
  rendered 400/500
  rendered 425/500
  rendered 450/500
  rendered 475/500
  rendered 500/500
[active] done 

## 6. Build per-state silent MP4 videos

Each silent video assembles the rendered PNG frames in order. The
duration of each frame is set to **exactly** the corresponding note's
duration + inter-note gap from the sonification mapping table. So the
total video duration equals the total audio duration, and they remain
sample-accurate when paired in the next cell.


In [ ]:
from moviepy.editor import (ImageClip, concatenate_videoclips, AudioFileClip,
                            VideoFileClip, clips_array, ColorClip,
                            CompositeVideoClip)

def build_silent_video(state, frames_subdir, mapping_csv, output_mp4):
    if not mapping_csv.exists():
        print(f'  [SKIP] {state}: no mapping CSV at {mapping_csv}')
        return None
    mapping = pd.read_csv(mapping_csv)

    clips = []
    for i, row in mapping.iterrows():
        if MAX_FRAMES is not None and i >= MAX_FRAMES:
            break
        frame_path = frames_subdir / f'frame_{i+1:04d}.png'
        if not frame_path.exists():
            continue
        clip_dur = float(row['duration_s']) + NOTE_GAP_S
        clips.append(ImageClip(str(frame_path)).set_duration(clip_dur))
    if not clips:
        print(f'  [SKIP] {state}: no frames matched mapping rows')
        return None

    video = concatenate_videoclips(clips, method='compose')
    video.write_videofile(str(output_mp4), fps=24, codec='libx264',
                          audio=False, preset='medium',
                          logger=None, verbose=False)
    print(f'  saved {output_mp4.name} ({video.duration:.1f} s)')
    return video.duration

silent_durations = {}
for state in SYSTEMS:
    out = VIDEO_DIR / f'mov_{state}_silent.mp4'
    dur = build_silent_video(
        state,
        FRAMES_DIR / state,
        PROCESSED_DIR / f'{state}_sonification_mapping.csv',
        out,
    )
    if dur is not None:
        silent_durations[state] = dur
print('\nSilent video durations (s):', silent_durations)

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



  saved mov_inactive_silent.mp4 (116.9 s)
  saved mov_active_silent.mp4 (167.4 s)
  saved mov_active_ligand_bound_silent.mp4 (119.7 s)

Silent video durations (s): {'inactive': np.float64(116.87002622876133), 'active': np.float64(167.43863389337903), 'active_ligand_bound': np.float64(119.69047873591845)}


## 7. Per-state per-instrument MP4 (with audio baked in)

For each `(state, instrument)` pair we combine the silent video with
the corresponding WAV produced by notebook 02. These nine files are
the standalone deliverables — drop them into a slide and they play
with sound on a single click.


In [ ]:
def combine_video_audio(silent_mp4, audio_wav, output_mp4):
    if not silent_mp4.exists() or not audio_wav.exists():
        print(f'  [SKIP] missing {silent_mp4.name} or {audio_wav.name}')
        return False
    video = VideoFileClip(str(silent_mp4))
    audio = AudioFileClip(str(audio_wav))
    final_duration = min(video.duration, audio.duration)
    video = video.subclip(0, final_duration)
    audio = audio.subclip(0, final_duration)
    final = video.set_audio(audio)
    final.write_videofile(str(output_mp4), fps=24, codec='libx264',
                          audio_codec='aac', preset='medium',
                          logger=None, verbose=False)
    video.close(); audio.close(); final.close()
    return True

for state in SYSTEMS:
    silent = VIDEO_DIR / f'mov_{state}_silent.mp4'
    for instrument in INSTRUMENTS:
        audio = AUDIO_DIR / f'{state}_{instrument}.wav'
        out   = VIDEO_DIR / f'mov_{state}_{instrument}.mp4'
        ok = combine_video_audio(silent, audio, out)
        if ok:
            print(f'  saved {out.name}')


  saved mov_inactive_piano.mp4
  saved mov_inactive_violin.mp4
  saved mov_inactive_flute.mp4
  saved mov_active_piano.mp4
  saved mov_active_violin.mp4
  saved mov_active_flute.mp4
  saved mov_active_ligand_bound_piano.mp4
  saved mov_active_ligand_bound_violin.mp4
  saved mov_active_ligand_bound_flute.mp4


## 8. Side-by-side pair MP4 (sequential A → B with pair audio)

For each `(pair, instrument)` we build a single MP4 whose frame is
horizontally split:

- **Left half** — the trajectory video for state A.
- **Right half** — the trajectory video for state B.

During the first part of the audio (state A's notes), the left panel
animates and the right panel is frozen on its first frame. During the
inter-state silence both are frozen. During the second part of the
audio, the right panel animates and the left panel holds on its last
frame. The total duration matches `pair_{pair}_{instrument}.wav` from
notebook 02.


In [ ]:
import imageio_ffmpeg
from moviepy.config import change_settings
# Point moviepy at imageio-ffmpeg's bundled binary. This is the only
# binary path that's guaranteed to exist after the pip install above,
# even on Colab runtimes where /usr/bin/ffmpeg is missing or stale.
change_settings({"FFMPEG_BINARY": imageio_ffmpeg.get_ffmpeg_exe()})

def extract_frame_to_image_clip(video_path, t):
    '''Grab a single frame at time `t` (seconds) from `video_path` and
    return it as a MoviePy ImageClip.

    Implementation: open the video with moviepy, ask its FFmpeg-backed
    reader for the frame as a numpy array via .get_frame(t), copy it
    out of the reader's internal buffer (the buffer is reused on the
    next read), close the reader. No temp files, no subprocess plumbing,
    no PIL round-trip.

    `t` is clamped into [0, duration - 1e-3] so we never request a frame
    past the end of the stream.'''
    with VideoFileClip(str(video_path)) as vid:
        t_safe = max(0.0, min(float(t), max(0.0, vid.duration - 1e-3)))
        frame = vid.get_frame(t_safe).copy()
    return ImageClip(frame)

def build_pair_video(pair_name, pair_cfg, instrument):
    state_a, state_b = pair_cfg['states']
    silent_a_path = VIDEO_DIR / f'mov_{state_a}_silent.mp4'
    silent_b_path = VIDEO_DIR / f'mov_{state_b}_silent.mp4'
    audio_pair    = AUDIO_DIR / f'pair_{pair_name}_{instrument}.wav'
    out           = VIDEO_DIR / f'mov_pair_{pair_name}_{instrument}.mp4'

    if not all(p.exists() for p in [silent_a_path, silent_b_path, audio_pair]):
        print(f'  [SKIP] {pair_name}/{instrument}: missing inputs')
        return False

    # Open the two silent state videos. We hold these open through the
    # whole composition because we need their VideoFileClip objects in
    # the concatenations below.
    vid_a = VideoFileClip(str(silent_a_path))
    vid_b = VideoFileClip(str(silent_b_path))
    duration_a, duration_b = vid_a.duration, vid_b.duration

    # Frozen frames for the "other panel" holds:
    #   - left panel freezes on state A's LAST frame after A finishes
    #   - right panel freezes on state B's FIRST frame until B starts
    frozen_a = extract_frame_to_image_clip(silent_a_path, duration_a - 0.1)
    frozen_b = extract_frame_to_image_clip(silent_b_path, 0.0)

    # Left panel: vid_a plays, then frozen-A holds through the inter-
    # state silence and through state B's playback duration.
    left = concatenate_videoclips([
        vid_a,
        frozen_a.set_duration(PAIR_SILENCE_S + duration_b),
    ])
    # Right panel: frozen-B holds through state A's playback duration
    # and the silence, then vid_b plays.
    right = concatenate_videoclips([
        frozen_b.set_duration(duration_a + PAIR_SILENCE_S),
        vid_b,
    ])

    total = min(left.duration, right.duration)
    left, right = left.subclip(0, total), right.subclip(0, total)

    combined = clips_array([[left, right]])
    audio = AudioFileClip(str(audio_pair))
    audio = audio.subclip(0, min(audio.duration, combined.duration))
    combined = combined.set_audio(audio)
    combined.write_videofile(str(out), fps=24, codec='libx264',
                             audio_codec='aac', preset='medium',
                             logger=None, verbose=False)

    for c in (vid_a, vid_b, left, right, combined, audio, frozen_a, frozen_b):
        try: c.close()
        except Exception: pass
    print(f'  saved {out.name}')
    return True

for pair_name, pair_cfg in PAIRS.items():
    for instrument in INSTRUMENTS:
        build_pair_video(pair_name, pair_cfg, instrument)


  warnings.warn("Warning: in file %s, "%(self.filename)+



  saved mov_pair_activation_piano.mp4
  saved mov_pair_activation_violin.mp4
  saved mov_pair_activation_flute.mp4


  warnings.warn("Warning: in file %s, "%(self.filename)+



  saved mov_pair_ligand_piano.mp4
  saved mov_pair_ligand_violin.mp4
  saved mov_pair_ligand_flute.mp4


In [ ]:
from moviepy.editor import VideoFileClip

# Path to the problematic video file
problem_video_path = VIDEO_DIR / 'mov_inactive_silent.mp4'

print(f"Attempting to load: {problem_video_path}")

try:
    # Try to load the video file
    clip = VideoFileClip(str(problem_video_path))
    print(f"Successfully loaded video. Duration: {clip.duration} seconds")
    clip.close()
except Exception as e:
    print(f"Failed to load video file. Error: {e}")
    print("This confirms the mov_inactive_silent.mp4 file itself is likely corrupted or unreadable by MoviePy.")


Attempting to load: /content/drive/MyDrive/GPCR_Sonification/outputs/video/mov_inactive_silent.mp4
Successfully loaded video. Duration: 116.88 seconds


## 9. Interactive HTML presentation player

This cell writes the HTML player in **two** locations:

1. `outputs/video/presentation.html` — references audio via
   `../audio/*.wav`. Works inside Colab (Drive is mounted, so the full
   `outputs/` tree is on disk) **only as long as the whole `outputs/`
   tree is available together** — if you download just the `.html`
   file, the relative `../audio/` path will 404 and the videos won't
   play.
2. `outputs/presentation_bundle/presentation.html` — a **fully
   self-contained folder**. The cell copies every silent MP4 and every
   per-state-per-instrument WAV into `outputs/presentation_bundle/`,
   and the HTML inside uses **flat sibling filenames only**. You can
   download this single folder, drop it on a USB stick, double-click
   the HTML on any laptop, and the videos play in sync with the audio
   — no server, no Colab, no Drive.

### Why the dual output

When the HTML lives on Drive and you download only the HTML, the
browser can't reach the videos and audio that are still on Drive (no
`file://` access). That is the failure mode that makes "videos don't
move" when opened locally. The bundle variant exists to make the
deliverable a single, portable folder.

### What you'll see in the HTML

One row per comparison pair. Within each row, two columns (state A on
the left, state B on the right). Each column has:

- the silent trajectory video at the top;
- three labelled `<audio controls>` bars stacked below
  (Piano, Violin, Flute).

Clicking play on any audio bar:
- pauses any other audio bar of the same panel,
- resets and plays the corresponding video in sync;
- when paused or seeked, the video follows.


In [ ]:
# ---------------------------------------------------------------------
# Build TWO presentation HTMLs:
#
#   (a) outputs/video/presentation.html
#       Uses '../audio/*.wav' relative paths. Works only when the whole
#       outputs/ tree is preserved (Drive-mounted Colab, or after
#       downloading outputs/ as a unit).
#
#   (b) outputs/presentation_bundle/presentation.html
#       SELF-CONTAINED folder. We copy each silent MP4 and each WAV
#       *into* outputs/presentation_bundle/ so every <video>/<audio>
#       src is just a flat sibling filename. You can zip this one
#       folder, download it, drop it on a USB stick, and the videos
#       play offline on any machine without a server.
#
# Background: a Drive-only HTML breaks the moment you copy just the
# .html file off Drive, because the <video src="mov_*.mp4"> and
# <audio src="../audio/*.wav"> targets are not on the local disk.
# The bundle variant fixes that.
# ---------------------------------------------------------------------
import shutil

BUNDLE_DIR = PROJECT_DIR / 'outputs' / 'presentation_bundle'
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

def _copy_if_newer(src, dst):
    '''Copy src -> dst unless dst already matches src by size+mtime.
    Avoids re-copying ~1 GB of MP4 on every re-run.'''
    if not src.exists():
        return False
    if dst.exists():
        s_st, d_st = src.stat(), dst.stat()
        if s_st.st_size == d_st.st_size and d_st.st_mtime >= s_st.st_mtime:
            return True
    shutil.copy2(src, dst)
    return True

def _html_for(asset_resolver):
    '''Build the full HTML string given an asset_resolver(kind, *args)
    that returns a relative path string usable in the HTML, or None if
    the asset isn't available.'''
    pair_blocks = []
    for pair_name, pair_cfg in PAIRS.items():
        cells = []
        for state in pair_cfg['states']:
            video_rel = asset_resolver('video', state)
            if video_rel is None:
                continue
            video_id  = f'vid-{pair_name}-{state}'
            audio_ids = {ins: f'aud-{pair_name}-{state}-{ins}' for ins in INSTRUMENTS}

            audio_rows = []
            for ins in INSTRUMENTS:
                audio_rel = asset_resolver('audio', state, ins)
                if audio_rel is None:
                    continue
                audio_rows.append(
                    f'        <div class="audio-row">\n'
                    f'          <span class="label">{ins.capitalize()}</span>\n'
                    f'          <audio id="{audio_ids[ins]}" controls preload="metadata"\n'
                    f'                 data-video="{video_id}" data-group="{pair_name}-{state}"\n'
                    f'                 src="{audio_rel}"></audio>\n'
                    f'        </div>'
                )
            audio_html = '\n'.join(audio_rows) if audio_rows else '        <p>(no audio)</p>'
            cells.append(
                f'      <div class="state-cell">\n'
                f'        <h3>{state}</h3>\n'
                # preload="auto" so the first frame is decoded and shown
                # immediately, instead of staying as a black rectangle
                # until the user starts the audio.
                f'        <video id="{video_id}" muted preload="auto" playsinline\n'
                f'               src="{video_rel}"></video>\n'
                f'{audio_html}\n'
                f'      </div>'
            )
        if cells:
            pair_blocks.append(
                f'    <section class="pair">\n'
                f'      <h2>{pair_cfg["label"]}</h2>\n'
                f'      <div class="state-row">\n'
                + '\n'.join(cells) + '\n'
                + '      </div>\n'
                f'    </section>'
            )

    pair_html = '\n'.join(pair_blocks)

    css = '''
    body { font-family: Helvetica, Arial, sans-serif; background:#f7f7f9; color:#222;
           margin: 24px; max-width: 1280px; }
    h1 { font-size: 22px; margin-bottom: 4px; }
    h2 { font-size: 16px; margin-top: 28px; border-bottom: 1px solid #ccc;
         padding-bottom: 4px; }
    h3 { font-size: 14px; margin: 0 0 6px 0; color: #444; text-transform: capitalize; }
    .state-row { display: flex; gap: 22px; flex-wrap: wrap; }
    .state-cell { flex: 1 1 360px; background: #fff; padding: 12px;
                  border: 1px solid #e2e2e6; border-radius: 6px; }
    video { width: 100%; height: auto; background: #000; border-radius: 4px;
            margin-bottom: 8px; }
    .audio-row { display: flex; align-items: center; gap: 10px;
                 margin: 4px 0; }
    .audio-row .label { display: inline-block; width: 60px; font-size: 13px;
                        color: #333; font-weight: 600; }
    .audio-row audio { flex: 1; height: 32px; }
    .legend { font-size: 12px; color: #555; margin-top: 6px; }
    .legend span { display: inline-block; margin-right: 12px; }
    .swatch { display:inline-block; width: 10px; height: 10px;
              border-radius: 2px; margin-right: 4px; vertical-align: middle; }
    '''

    js = '''
    document.querySelectorAll('audio').forEach(function (audio) {
      var video = document.getElementById(audio.dataset.video);
      var group = audio.dataset.group;
      audio.addEventListener('play', function () {
        // Pause any other audio in the same group (same state panel)
        document.querySelectorAll('audio[data-group="' + group + '"]').forEach(function (a) {
          if (a !== audio && !a.paused) a.pause();
        });
        if (video) {
          video.currentTime = audio.currentTime;
          video.play().catch(function(){});
        }
      });
      audio.addEventListener('pause', function () {
        if (video) video.pause();
      });
      audio.addEventListener('seeked', function () {
        if (video) video.currentTime = audio.currentTime;
      });
      audio.addEventListener('ended', function () {
        if (video) video.pause();
      });
    });
    '''

    return (
        '<!DOCTYPE html>\n<html lang="en">\n<head>\n'
        '<meta charset="utf-8">\n'
        '<title>β2AR sonification - presentation</title>\n'
        '<style>\n' + css + '\n</style>\n</head>\n<body>\n'
        '<h1>Listening to GPCR Dynamics — β2AR sonification</h1>\n'
        '<p>Click any audio bar to play that instrument; the protein '
        'trajectory above plays in sync. Pause to freeze the protein at '
        'that moment.</p>\n'
        '<div class="legend">\n'
        '  <span><span class="swatch" style="background:#4a90e2"></span>protein cartoon</span>\n'
        '  <span><span class="swatch" style="background:#d33"></span>TM3 intracellular</span>\n'
        '  <span><span class="swatch" style="background:#e69f00"></span>TM6 intracellular</span>\n'
        '  <span><span class="swatch" style="background:#8e44ad"></span>NPxxY motif</span>\n'
        '  <span><span class="swatch" style="background:#f1c40f"></span>DRY ionic lock</span>\n'
        '  <span><span class="swatch" style="background:#2e8b57"></span>ligand</span>\n'
        '</div>\n'
        + pair_html + '\n'
        '<script>\n' + js + '\n</script>\n'
        '</body>\n</html>\n'
    )

# ---------- Variant (a): in-place HTML at outputs/video/ ----------
def drive_resolver(kind, *args):
    if kind == 'video':
        state = args[0]
        src = VIDEO_DIR / f'mov_{state}_silent.mp4'
        return f'mov_{state}_silent.mp4' if src.exists() else None
    if kind == 'audio':
        state, ins = args
        src = AUDIO_DIR / f'{state}_{ins}.wav'
        return f'../audio/{state}_{ins}.wav' if src.exists() else None
    return None

drive_html_path = VIDEO_DIR / 'presentation.html'
drive_html_path.write_text(_html_for(drive_resolver), encoding='utf-8')
print(f'Saved {drive_html_path}  ({drive_html_path.stat().st_size // 1024} kB)')

# ---------- Variant (b): self-contained bundle folder ----------
print(f'\nBuilding self-contained bundle at: {BUNDLE_DIR}')
copied = 0
for state in SYSTEMS:
    src = VIDEO_DIR / f'mov_{state}_silent.mp4'
    dst = BUNDLE_DIR / src.name
    if _copy_if_newer(src, dst):
        copied += 1
    for ins in INSTRUMENTS:
        src = AUDIO_DIR / f'{state}_{ins}.wav'
        dst = BUNDLE_DIR / src.name
        if _copy_if_newer(src, dst):
            copied += 1

def bundle_resolver(kind, *args):
    if kind == 'video':
        state = args[0]
        name = f'mov_{state}_silent.mp4'
        return name if (BUNDLE_DIR / name).exists() else None
    if kind == 'audio':
        state, ins = args
        name = f'{state}_{ins}.wav'
        return name if (BUNDLE_DIR / name).exists() else None
    return None

bundle_html_path = BUNDLE_DIR / 'presentation.html'
bundle_html_path.write_text(_html_for(bundle_resolver), encoding='utf-8')

bundle_size_mb = sum(f.stat().st_size for f in BUNDLE_DIR.iterdir()
                     if f.is_file()) / (1024 * 1024)
print(f'  copied {copied} asset files into the bundle')
print(f'  saved  {bundle_html_path.name}')
print(f'  bundle size: {bundle_size_mb:.1f} MB')
print(f'\nTo present offline: download the WHOLE folder')
print(f'    {BUNDLE_DIR}')
print(f'as a unit, then double-click presentation.html on your laptop.')


Saved /content/drive/MyDrive/GPCR_Sonification/outputs/video/presentation.html  (7 kB)

Building self-contained bundle at: /content/drive/MyDrive/GPCR_Sonification/outputs/presentation_bundle
  copied 12 asset files into the bundle
  saved  presentation.html
  bundle size: 214.2 MB

To present offline: download the WHOLE folder
    /content/drive/MyDrive/GPCR_Sonification/outputs/presentation_bundle
as a unit, then double-click presentation.html on your laptop.
